In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

## RAG절차

- 1.docx문서를 읽는다(python-docx) : \n은 제외
- 2.읽어온 문서를 쪼갠다.(tiktoken이용)
    * 모델의 context window와 비용확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임배딩 -> vector Store에 저장 => chroma(local파일)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성(LangChain)

# 0. 패키지 설치

In [2]:
# 문서읽어오기
# pip install -q doc2txt

In [9]:
# text를 chunk로 나누는 기능(cf.)
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 파일저장 chroma(cf. chromadb와다름)
%pip install -q langchain-chroma

In [7]:
# langchain사용
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.


# 1. 문서읽기 (x)

In [5]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tex_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 3.69 s
Wall time: 3.76 s


1

# 2. 문서를 쪼개면서 읽기(o)
## 2.1 1500토큰씩 겹처서 쪼개면서읽기

In [6]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./tex_docx/소득세법(법률)(제21221호)(20260701).docx')
# gpt-4o-mini 모델의 토큰 추출방식 : cl100k_base 방식
# gpt-4 모델의 토큰 추출방식 : o200k_base방식
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", # 토큰을 세는 방식 이름
    chunk_size=1500,             # 1500토큰 기준으로 chunk 분리
    chunk_overlap=200,
    # separator 파라미터가 없어
)
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 겹쳐 쪼개면서 읽어오는 시간:',runtime)
print('chunk갯수:',len(documents))

문서를 겹쳐 쪼개면서 읽어오는 시간: 4.4544517993927
chunk갯수: 180


In [7]:
# 각 chunk별 글자수
print([len(document.page_content) for document in documents])

[1718, 1657, 1639, 1649, 1738, 1455, 1280, 1534, 1326, 1613, 1596, 1590, 1575, 1633, 1622, 1561, 1615, 1643, 1574, 1475, 1423, 1607, 1450, 1500, 1641, 1602, 1538, 1649, 1672, 1595, 1603, 1671, 1588, 1636, 1599, 1542, 1564, 1600, 1697, 1716, 1660, 1632, 1632, 1656, 1664, 1604, 1484, 1670, 1704, 1647, 1584, 1528, 1625, 1625, 1636, 1621, 1575, 1625, 1600, 1544, 1578, 1612, 1604, 1672, 1692, 1639, 1669, 1624, 1626, 1627, 1615, 1611, 1608, 1681, 1702, 1689, 1605, 1541, 1570, 1733, 2048, 1592, 1629, 1582, 1616, 1605, 1622, 1612, 1596, 1674, 1693, 1720, 1657, 1706, 1527, 1552, 1584, 1610, 1675, 1693, 1669, 1623, 1569, 1715, 1689, 1629, 1554, 1623, 1691, 1677, 1683, 1615, 1657, 1591, 1482, 1531, 1589, 1690, 1568, 1527, 1449, 1577, 1555, 1444, 1540, 1536, 1611, 1684, 1711, 1667, 1646, 1621, 1742, 1583, 1634, 1623, 1613, 1662, 1560, 1632, 1543, 1645, 1529, 1535, 1603, 1595, 1615, 1641, 1556, 1512, 1685, 1629, 1612, 1593, 1612, 1576, 1666, 1604, 1629, 1575, 1602, 1614, 1516, 1569, 1587, 1608, 162

In [8]:
documents[0].page_content

'소득세법\n\n\n\n소득세법\n\n[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]\n\n재정경제부(재산세제과(양도소득세)) 044-215-4312\n\n재정경제부(소득세제과(근로소득)) 044-215-4216\n\n재정경제부(금융세제과(이자소득, 배당소득)) 044-215-4233\n\n재정경제부(소득세제과(사업소득, 기타소득)) 044-215-4217\n\n재정경제부(국제조세제도과(비거주자)) 044-215-4651\n\n\n\n제1장 총칙 <개정 2009. 12. 31.>\n\n\n\n제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.\n\n[본조신설 2009. 12. 31.]\n\n[종전 제1조는 제2조로 이동 <2009. 12. 31.>]\n\n\n\n제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.>\n\n1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다.\n\n2. “비거주자”란 거주자가 아닌 개인을 말한다.\n\n3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.\n\n4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다.\n\n5. “사업자”란 사업소득이 있는 거주자를 말한다.\n\n② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.\n\n[본조신설 2009. 12. 31.]\n\n\n\n제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다.\n\n1. 거주자\n\n2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인\n\n② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수

In [9]:
documents[1].page_content

' 이익의 분배비율이 확인되는 경우\n\n2. 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 것으로 확인되는 경우\n\n④ 제3항에도 불구하고 해당 단체의 전체 구성원 중 일부 구성원의 분배비율만 확인되거나 일부 구성원에게만 이익이 분배되는 것으로 확인되는 경우에는 다음 각 호의 구분에 따라 소득세 또는 법인세를 납부할 의무를 진다.<신설 2018. 12. 31.>\n\n1. 확인되는 부분: 해당 구성원별로 소득세 또는 법인세에 대한 납세의무 부담\n\n2. 확인되지 아니하는 부분: 해당 단체를 1거주자 또는 1비거주자로 보아 소득세에 대한 납세의무 부담\n\n⑤ 제3항 및 제4항에도 불구하고 법인으로 보는 단체 외의 법인 아닌 단체에 해당하는 국외투자기구(투자권유를 하여 모은 금전 등을 가지고 재산적 가치가 있는 투자대상자산을 취득, 처분하거나 그 밖의 방법으로 운용하고 그 결과를 투자자에게 배분하여 귀속시키는 투자행위를 하는 기구로서 국외에서 설립된 기구를 말한다. 이하 같다)를 제119조의2제1항제2호에 따라 국내원천소득의 실질귀속자로 보는 경우 그 국외투자기구는 1비거주자로서 소득세를 납부할 의무를 진다.<신설 2018. 12. 31.>\n\n[전문개정 2009. 12. 31.]\n\n[제1조에서 이동, 종전 제2조는 제2조의2로 이동 <2009. 12. 31.>]\n\n\n\n제2조의2(납세의무의 범위) ① 제43조에 따라 공동사업에 관한 소득금액을 계산하는 경우에는 해당 공동사업자별로 납세의무를 진다. 다만, 제43조제3항에 따른 주된 공동사업자(이하 이 항에서 “주된 공동사업자”라 한다)에게 합산과세되는 경우 그 합산과세되는 소득금액에 대해서는 주된 공동사업자의 특수관계인은 같은 조 제2항에 따른 손익분배비율에 해당하는 그의 소득금액을 한도로 주된 공동사업자와 연대하여 납세의무를 진다. <개정 2012. 1. 1., 2013. 1. 1.>\n\n② 제44조에 따라 피상속인의 소득금액에 대해서 과세하는 경우에는

In [10]:
documents[2].page_content

' 신탁의 이익을 받을 수익자(수익자가 사망하는 경우에는 그 상속인)에게 귀속되는 것으로 본다.\n\n② 제1항에도 불구하고 위탁자가 신탁재산을 실질적으로 통제하는 등 대통령령으로 정하는 요건을 충족하는 신탁의 경우에는 그 신탁재산에 귀속되는 소득은 위탁자에게 귀속되는 것으로 본다.<개정 2023. 12. 31.>\n\n[본조신설 2020. 12. 29.]\n\n\n\n제3조(과세소득의 범위) ① 거주자에게는 이 법에서 규정하는 모든 소득에 대해서 과세한다. 다만, 해당 과세기간 종료일 10년 전부터 국내에 주소나 거소를 둔 기간의 합계가 5년 이하인 외국인 거주자에게는 과세대상 소득 중 국외에서 발생한 소득의 경우 국내에서 지급되거나 국내로 송금된 소득에 대해서만 과세한다.\n\n② 비거주자에게는 제119조에 따른 국내원천소득에 대해서만과세한다.\n\n③ 제1항 및 제2항을 적용하는 경우 「조세특례제한법」 제100조의14제2호의 동업자에게는 같은 법 제100조의18제1항에 따라 배분받은 소득 및 같은 법 제100조의22제1항에 따라 분배받은 자산의 시가 중 분배일의 지분가액을 초과하여 발생하는 소득에 대하여 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제4조(소득의 구분) ① 거주자의 소득은 다음 각 호와 같이 구분한다. <개정 2013. 1. 1., 2020. 12. 29., 2024. 12. 31.>\n\n1. 종합소득\n\n\u3000이 법에 따라 과세되는 모든 소득에서 제2호 및 제3호에 따른 소득을 제외한 소득으로서 다음 각 목의 소득을 합산한 것\n\n가. 이자소득\n\n나. 배당소득\n\n다. 사업소득\n\n라. 근로소득\n\n마. 연금소득\n\n바. 기타소득\n\n2. 퇴직소득\n\n2의2. 삭제<2024. 12. 31.>\n\n3. 양도소득\n\n② 제1항에 따른 소득을 구분할 때 다음 각 호의 신탁을 제외한 신탁의 이익은 「신탁법」 제2조에 따라 수탁자에게 이전되거나 그 밖에 처분된 재산권에서 발생하는 소득의 내용별로 

In [11]:
documents[3].page_content

'외로 이전(이하 “출국”이라 한다)하여 비거주자가 되는 경우의 과세기간은 1월 1일부터 출국한 날까지로 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제6조(납세지) ① 거주자의 소득세 납세지는 그 주소지로 한다. 다만, 주소지가 없는 경우에는 그 거소지로 한다.\n\n② 비거주자의 소득세 납세지는 제120조에 따른 국내사업장(이하 “국내사업장”이라 한다)의 소재지로 한다. 다만, 국내사업장이 둘 이상 있는 경우에는 주된 국내사업장의 소재지로 하고, 국내사업장이 없는 경우에는 국내원천소득이 발생하는 장소로 한다.<개정 2013. 1. 1.>\n\n③ 납세지가 불분명한 경우에는 대통령령으로 정하는 바에 따라 납세지를 결정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제7조(원천징수 등의 경우의 납세지) ① 원천징수하는 소득세의 납세지는 다음 각 호에 따른다. <개정 2012. 1. 1., 2023. 12. 31.>\n\n1. 원천징수하는 자가 거주자인 경우: 그 거주자의 주된 사업장 소재지. 다만, 주된 사업장 외의 사업장에서 원천징수를 하는 경우에는 그 사업장의 소재지, 사업장이 없는 경우에는 그 거주자의 주소지 또는 거소지로 한다.\n\n2. 원천징수하는 자가 비거주자인 경우: 그 비거주자의 주된 국내사업장 소재지. 다만, 주된 국내사업장 외의 국내사업장에서 원천징수를 하는 경우에는 그 국내사업장의 소재지, 국내사업장이 없는 경우에는 그 비거주자의 거류지(居留地) 또는 체류지로 한다.\n\n3. 원천징수하는 자가 법인인 경우: 그 법인의 본점 또는 주사무소의 소재지\n\n4. 원천징수하는 자가 법인인 경우로서 그 법인의 지점, 영업소, 그 밖의 사업장이 독립채산제(獨立採算制)에 따라 독자적으로 회계사무를 처리하는 경우: 제3호에도 불구하고 그 사업장의 소재지(그 사업장의 소재지가 국외에 있는 경우는 제외한다). 다만, 대통령령으로 정하는 경우에는 그 법인의 본점 또는 주사무소의 소재지를 소득세 원천징수세액의 납세지로 할 수 

In [12]:
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents]))

1611.5555555555557
2048
1280


## 2.2 1500글자씩 겹쳐 쪼개 읽어오기

In [13]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=200, # 200문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 :',runtime)
print('chunk갯수:',len(documents))

문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 : 3.396960973739624
chunk갯수: 195


In [14]:
documents[2].page_content

'⑤ 공동으로 소유한 자산에 대한 양도소득금액을 계산하는 경우에는 해당 자산을 공동으로 소유하는 각 거주자가 납세의무를 진다.<신설 2017. 12. 19., 2020. 12. 29.>\n\n[전문개정 2009. 12. 31.]\n\n[제2조에서 이동 <2009. 12. 31.>]\n\n\n\n제2조의3(신탁재산 귀속 소득에 대한 납세의무의 범위) ① 신탁재산에 귀속되는 소득은 그 신탁의 이익을 받을 수익자(수익자가 사망하는 경우에는 그 상속인)에게 귀속되는 것으로 본다.\n\n② 제1항에도 불구하고 위탁자가 신탁재산을 실질적으로 통제하는 등 대통령령으로 정하는 요건을 충족하는 신탁의 경우에는 그 신탁재산에 귀속되는 소득은 위탁자에게 귀속되는 것으로 본다.<개정 2023. 12. 31.>\n\n[본조신설 2020. 12. 29.]\n\n\n\n제3조(과세소득의 범위) ① 거주자에게는 이 법에서 규정하는 모든 소득에 대해서 과세한다. 다만, 해당 과세기간 종료일 10년 전부터 국내에 주소나 거소를 둔 기간의 합계가 5년 이하인 외국인 거주자에게는 과세대상 소득 중 국외에서 발생한 소득의 경우 국내에서 지급되거나 국내로 송금된 소득에 대해서만 과세한다.\n\n② 비거주자에게는 제119조에 따른 국내원천소득에 대해서만과세한다.\n\n③ 제1항 및 제2항을 적용하는 경우 「조세특례제한법」 제100조의14제2호의 동업자에게는 같은 법 제100조의18제1항에 따라 배분받은 소득 및 같은 법 제100조의22제1항에 따라 분배받은 자산의 시가 중 분배일의 지분가액을 초과하여 발생하는 소득에 대하여 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제4조(소득의 구분) ① 거주자의 소득은 다음 각 호와 같이 구분한다. <개정 2013. 1. 1., 2020. 12. 29., 2024. 12. 31.>\n\n1. 종합소득\n\n\u3000이 법에 따라 과세되는 모든 소득에서 제2호 및 제3호에 따른 소득을 제외한 소득으로서 다음 각 목의 소득을 합산한 것\n\n

In [15]:
documents[1].page_content

'1. 구성원 간 이익의 분배비율이 정하여져 있고 해당 구성원별로 이익의 분배비율이 확인되는 경우\n\n2. 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 것으로 확인되는 경우\n\n④ 제3항에도 불구하고 해당 단체의 전체 구성원 중 일부 구성원의 분배비율만 확인되거나 일부 구성원에게만 이익이 분배되는 것으로 확인되는 경우에는 다음 각 호의 구분에 따라 소득세 또는 법인세를 납부할 의무를 진다.<신설 2018. 12. 31.>\n\n1. 확인되는 부분: 해당 구성원별로 소득세 또는 법인세에 대한 납세의무 부담\n\n2. 확인되지 아니하는 부분: 해당 단체를 1거주자 또는 1비거주자로 보아 소득세에 대한 납세의무 부담\n\n⑤ 제3항 및 제4항에도 불구하고 법인으로 보는 단체 외의 법인 아닌 단체에 해당하는 국외투자기구(투자권유를 하여 모은 금전 등을 가지고 재산적 가치가 있는 투자대상자산을 취득, 처분하거나 그 밖의 방법으로 운용하고 그 결과를 투자자에게 배분하여 귀속시키는 투자행위를 하는 기구로서 국외에서 설립된 기구를 말한다. 이하 같다)를 제119조의2제1항제2호에 따라 국내원천소득의 실질귀속자로 보는 경우 그 국외투자기구는 1비거주자로서 소득세를 납부할 의무를 진다.<신설 2018. 12. 31.>\n\n[전문개정 2009. 12. 31.]\n\n[제1조에서 이동, 종전 제2조는 제2조의2로 이동 <2009. 12. 31.>]\n\n\n\n제2조의2(납세의무의 범위) ① 제43조에 따라 공동사업에 관한 소득금액을 계산하는 경우에는 해당 공동사업자별로 납세의무를 진다. 다만, 제43조제3항에 따른 주된 공동사업자(이하 이 항에서 “주된 공동사업자”라 한다)에게 합산과세되는 경우 그 합산과세되는 소득금액에 대해서는 주된 공동사업자의 특수관계인은 같은 조 제2항에 따른 손익분배비율에 해당하는 그의 소득금액을 한도로 주된 공동사업자와 연대하여 납세의무를 진다. <개정 2012. 1. 1., 2013. 1. 1.>\n\n②

In [16]:
# chunk들의 글자수
print([len(document.page_content)for document in documents])

[1446, 1421, 1482, 1487, 1479, 1408, 1457, 1495, 1467, 1446, 1487, 1456, 1467, 1351, 1392, 1362, 1402, 1470, 1410, 1489, 1455, 1496, 1441, 1319, 1458, 1476, 1452, 1382, 1384, 1467, 1227, 1494, 1494, 1470, 1454, 1495, 1412, 1477, 1477, 1362, 1449, 1386, 1055, 1467, 1361, 1493, 1467, 1434, 1377, 1468, 1494, 1457, 1442, 1370, 873, 1419, 1357, 1353, 1316, 1349, 1452, 1439, 1363, 1433, 1412, 1306, 1200, 1411, 1452, 1421, 1318, 1416, 1333, 1308, 1385, 1479, 1495, 1399, 1375, 1360, 1353, 1382, 1446, 1356, 1409, 1483, 1486, 1157, 1454, 1407, 1186, 1376, 1489, 1487, 1444, 928, 1451, 1272, 1352, 1477, 1444, 1498, 1448, 1436, 1322, 1462, 1435, 1236, 1477, 1396, 1423, 1282, 1496, 1486, 1376, 1342, 1466, 1385, 1491, 1477, 1470, 1385, 1477, 1445, 1485, 1373, 1495, 1443, 1419, 1456, 1451, 1305, 1454, 1411, 1443, 1488, 1404, 1419, 1339, 1451, 1288, 1450, 1481, 1419, 1369, 1479, 1480, 1461, 1414, 1419, 1471, 1475, 1498, 1425, 1251, 1327, 1429, 1403, 1457, 1345, 1399, 1483, 1207, 1449, 1470, 1363, 1469,

# 3. 쪼갠 문서를 임배딩 ->벡터 데이터 베이스(파일)에 저장
- 임베딩 모델 : text-embedding-3-large (기본모델:text-embedding-ada-002)
- vector DB 저장 폴더 : ./chroma

In [17]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embeding = OpenAIEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'text-embedding-3-large'
)

In [18]:
# embed_query() : 한 문자욜 임베딩 벡터로 전환
len(embeding.embed_query("소득세법은 어쩌구 저쩌구"))

3072

In [19]:
# embed_document() : 여러 문자열을 임베딩 백터로 전환
vectors = embeding.embed_documents([
    documents[0].page_content, documents[1].page_content
])


In [ ]:
print(len(vector[0]),len(vector[1]))
print

In [23]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할때
# database = Chroma.from_documents(
#     documents=documents,# chunking 작업한 documents
#     embedding=embeding,# 임베딩 객체
#     collection_name='tax-collection', # 생략시 이름이 랜덤
#     persist_directory='./chroma'      # 생략시 로컬에 저장 안 됨. 인메모리 vector DB
# )
# 저장된 vector DB를 가져올 때
database = Chroma(
    embedding_function=embeding,
    collection_name='tax-collection',
    persist_directory='./chroma'
)

CPU times: total: 15.6 ms
Wall time: 14.8 ms


In [21]:
database.similarity_search(query="연봉 5천만원인 직장인의 소득세는?")

[Document(id='b478fb4a-3608-420b-90d6-0f01d656da91', metadata={'source': './tex_docx/소득세법(법률)(제21221호)(20260701).docx'}, page_content='[전문개정 2009. 12. 31.]\n\n\n\n제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>\n\n\n\n제1절 비과세 <개정 2009. 12. 31.>\n\n\n\n제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>\n\n1. 「공익신탁법」에 따른 공익신탁의 이익\n\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\

In [26]:
# 확인
results = database.get(include=['embeddings','documents','metadatas'])
print('데이터수:',len(results['ids']))
print('문서 임베딩 차원 수 :', len(results['embeddings'][0]))
print('1번째 원본:',results['documents'][1][:100])
print('1번째 원본의 임베딩 벡터 :',results['embeddings'][1][:10])
print('1번째 metabase:',results['meta'])

데이터수: 195


# 4.vector DB에 질문과 유사도 검색(retrieval)

In [27]:
query = "연봉 5천만원인 직장인의 소득세는 얼마에요?"
retrieved_docs = database.similarity_search(query=query,
                                           k=2)# 기본 k값은 4

In [29]:
len(retrieved_docs)

2

In [33]:
lst = ['a','b']
print('\n--\n'.join(lst))

a
--
b


In [35]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변생성 -1

In [36]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm = ChatOpenAI(model='gpt-4.1-nano')

In [37]:
prompt = f"""[identity]
-당신은 최고의 한국 소득세법 전문가입니다.
-[context]를 참조해서 사용자의 질문에 답변해주세요. context는 다음과 같습니다.
{retrieved_doc}
질문 : {query}
"""
ai_message=llm.invoke(prompt)

In [38]:
ai_message.usage_metadata

{'input_tokens': 1943,
 'output_tokens': 659,
 'total_tokens': 2602,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [39]:
print(ai_message.content)

연봉 5천만 원인 직장인의 소득세를 계산하기 위해서는 우선 근로소득금액과 이에 대한 근로소득세율을 적용해야 합니다. 다음은 기본적인 계산 과정입니다.

1. **총급여액**: 50,000,000원 (연봉)
2. **근로소득공제**: 근로소득공제는 근로소득자의 필요경비로, 법에 따라 일정 금액이 공제됩니다. 2024년 기준으로 근로소득공제액은 다음과 같습니다.

| 연봉 구간 | 근로소득공제액 (원) |
|--------------|-------------------|
| 1,200만원 이하 | 연봉의 70% (단, 한도 1200만원) | → 5,000만원은 1200만원 한도 적용, 공제액은 1,200만원
| 1,200만원 초과 | 아래 공식을 적용합니다.  |

그러나 간단한 계산을 위해 일반적인 예상 세율과 공제액을 적용해서 계산하면,

- **근로소득공제**: 5,000만원의 경우, 정부의 근로소득공제 계산 방식에 따라 약 1,200만원 공제
- **과세표준**: 50,000,000원 - 1,200만원 = 38,800,000원

3. **근로소득세**:
한국의 2024년 근로소득세율은 대략 다음과 같이 진행됩니다.
- 과세표준 1,200만원 이하: 세율 6%
- 1,200만원 초과 4,600만원 이하: 15%
- 4,600만원 초과 8,800만원 이하: 24%
- 8,800만원 초과 1억 5천만원 이하: 35%
- 그 이상: 38%

우리의 과세표준은 약 3,880만원이므로, 1,200만원에 6%, 나머지 2,680만원에 15%를 적용합니다.

계산:
- 1,200만원 × 6% = 72만원
- (3,880만원 - 1,200만원) = 2,680만원 × 15% = 약 402만원

즉, 근로소득세는 대략 **약 474만원** 내외가 예상됩니다.

단, 이 금액은 표준적인 계산에 근거한 대략적인 것으로, 정확한 세액은 기타 공제(인적공제, 보험료공제 등)와 지방세를 고려해서 세무서 또는 국세청 자료를 통해 계산하는 것이 필요합니다.

---

**요약:**  
연

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변생성 -2


In [40]:
from langchain_core.prompts import ChatPromptTemplate
promptTemplate = ChatPromptTemplate([
    ("system", "당신은 최고의 한국 소득세법 전문가입니다."),
    ("human", f"""다음 문맥을 참고하여 질문에 답변하세요. 답을 모른다면 모른다고 하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : """)
])
promptTemplate

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 최고의 한국 소득세법 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 문맥을 참고하여 질문에 답변하세요. 답을 모른다면 모른다고 하세요.\n    최대 3문장으로 간결하게 답변하세요.\n    질문 : {question}\n    문맥 : {context}\n    답변 : '), additional_kwargs={})])

In [41]:
prompt = promptTemplate.invoke({'context':retrieved_doc,
                               'question':query})

In [42]:
ai_message = llm.invoke(prompt) # 과금

In [43]:
ai_message.content

'연봉 5천만원인 직장인의 소득세는 개인의 공제 조건과 세율에 따라 차이가 있지만, 대략적으로 1,000만 원 내외가 예상됩니다. 이 계산은 기본적인 근로소득에 대한 소득세율과 공제를 고려한 추정치입니다. 정확한 금액은 정확한 공제액과 세율 적용 후 산출되어야 합니다.'

In [45]:
from langchain_core.output_parsers import StrOutputParser
outputParsers = StrOutputParser()
outputParsers.invoke(ai_message)

'연봉 5천만원인 직장인의 소득세는 개인의 공제 조건과 세율에 따라 차이가 있지만, 대략적으로 1,000만 원 내외가 예상됩니다. 이 계산은 기본적인 근로소득에 대한 소득세율과 공제를 고려한 추정치입니다. 정확한 금액은 정확한 공제액과 세율 적용 후 산출되어야 합니다.'

In [46]:
#위의 작업을 한눈에
outputParsers.invoke(llm.invoke(promptTemplate.invoke(
    {'context':retrieved_doc,'question':query}
)))

'한국 소득세법에 따르면, 연봉 5천만원인 직장인의 소득세는 구체적인 공제 및 세액 공제 내용에 따라 달라집니다. 일반적으로 근로소득에 대한 표준세율을 적용하면 약 1,000만원 내외의 세금이 부과될 수 있으나, 정확한 금액을 산출하려면 공제 및 세액 공제 내역이 필요합니다. 따라서 구체적인 납세액은 개별 조건에 따라 차이가 있을 수 있습니다.'

# 6. langchain으로 답변 생성

In [47]:
rag_chain = promptTemplate | llm | outputParsers
rag_chain.invoke({'context':retrieved_doc,'question':query})

'연봉 5천만원인 직장인의 소득세는 개인의 구체적인 근로소득공제, 기타 과세 대상 소득 여부에 따라 달라지지만, 대략적으로 1,000만 원에서 1,200만 원 정도일 수 있습니다. 정확한 산출을 위해서는 세율 적용과 공제 내역을 반드시 검토해야 합니다.'

Langchain으로 전달

In [57]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임배딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-large")
llm = ChatOpenAI(model='gpt-4.1-mini')

# 2. vector DB load
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma',
    collection_name='tax-collection'
)

# 3. Retriever 생성
# (1)search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity',
                                    search_kwarge={'k':3}) # 유사도가 높은 top-3추출
# (2)search_type='mmr':
# retriever = vectorStore.as_retriever(search_type='mmr',
#                                     search_kwarge={'k':3,
#                                                   'fetch_k':10,
#                                                   'lambda_mult':0.5})# 1에 가까울수록 관련성 깊은 계산
# (3)search_type='similarity_score_threshold' :
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold',
#                                     search_kwarge={'k':3,
#                                                   'score_threshold':0.4})

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다.다음 문맥을 참고하여 질문에 답하세요. 
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)

# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])


In [59]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'연봉 5,000만원인 직장인의 소득세는 근로소득공제와 기본공제, 세율 등을 적용해야 산출되나, 구체적인 공제내역과 세율 정보가 주어지지 않아 정확한 금액 산출이 어렵습니다. 일반적으로 2024년 기준 근로소득세율에 따라 약 250만~350만원 수준으로 추정되나 개인별 공제 상황에 따라 달라집니다. 정확한 계산을 위해서는 신고서 작성 시 자세한 공제항목과 세율표를 참고해야 합니다.'